In [5]:
import pandas as pd
import re
import numpy as np
import click
from datetime import datetime

from BuildDryLabMetricsTable import compute_drylab_metrics_table
from PlotMetricsPerCohort import *
from utils_definitions import *

In [6]:
wetlab_qc_metrics_file = '/home/rchamorro/projects/Protocols_metrics/wetdry_metrics/templates/template_WetLab_QC_metrics_Duplex_library.xlsx'

In [7]:
qc_wetlab_metrics = pd.read_excel(wetlab_qc_metrics_file, skiprows=0)
qc_wetlab_metrics

,BATCH,WetLab ID,Panel,Number of captures,Panel Size (Kb),Input (ng),DNA after ligation (ng),fmol to PCR1,DryLab ID
0,IDT014,IDT014_01,Pancancer,1,800,250,47.2,28.40,IA012_C_1_H_1
1,IDT014,IDT014_02,Pancancer,1,800,250,57.6,21.80,IA012_C_2_H_1
2,IDT014,IDT014_03,Pancancer,1,800,250,62.8,23.70,IA013_C_1_H_1
3,IDT014,IDT014_04,Pancancer,1,800,250,50.2,14.00,IA013_C_2_H_1
4,IDT014,IDT014_05,Pancancer,1,800,250,53.0,14.00,IA014_C_1_H_1
5,IDT014,IDT014_06,Pancancer,1,800,250,58.2,21.50,IA014_C_2_H_1
6,IDT014,IDT014_07,Pancancer,1,800,250,62.0,16.20,IA017_C_1_H_1
7,IDT014,IDT014_08,Pancancer,1,800,250,46.8,23.60,IA017_C_2_H_1
8,IDT018,IDT018_01,Pancancer,1,800,250,58.2,48.50,B_4_1_H_1
9,IDT018,IDT018_02,Pancancer,1,800,250,63.8,41.00,B_4_1_H_2


In [ ]:
qc_wetlab_metrics = pd.read_excel(wetlab_qc_metrics_file, skiprows=1)

# Fix column names

subheadings = list(qc_metrics.iloc[:1].fillna('').values[0])
column_names = list(qc_metrics.columns)

renamed_cols = []

for col in column_names:
    if col.endswith(".1"):
        renamed_cols.append(col.rstrip(".1"))
    elif 'Unnamed' not in col:
        renamed_cols.append(col)
    else:
        renamed_cols.append(re.sub(r"Unnamed:\s+\d+", "", col))
        
final_headers = []
replace_cols_dict = {}

for i in range(len(renamed_cols)):
    if renamed_cols[i] == '':
        final_headers.append(subheadings[i])
        replace_cols_dict[column_names[i]] = subheadings[i]
    else:
        if subheadings[i] == '':
            final_headers.append(renamed_cols[i])
            replace_cols_dict[column_names[i]] = renamed_cols[i]
        else:
            final_headers.append(renamed_cols[i] + ' --> ' + subheadings[i])
            replace_cols_dict[column_names[i]] = renamed_cols[i] + ' --> ' + subheadings[i]
            
qc_metrics = qc_metrics.drop([0]).rename(columns=replace_cols_dict).reset_index(drop = True)

cols2change = ['Panel Size (Kb)','Quantification gDNA --> Qubit conc. (ng/ul)',
            'DNA integrity --> DIN','Input (ng)','TapeStation (after fragmentation) --> Size (bp)',
            'TapeStation (after ligation) --> Size (bp)', 'Quantification (after ligation) --> DNA amount (ng)',
            'qPCR --> Unique DNA amount (fmol)','Real diversity --> fmol to PCR1',
            'Quantification POST-PCR1 --> DNA amount (ng)',
            'Real input to capture --> DNA amount (ng)',
            'TapeStation after PCR1 --> Size (bp)',
            'Quantification POST-PCR2 --> Qubit conc. (ng/ul)', 'Quantification POST-PCR2 --> DNA amount (ng)',
            'Quantification POST-PCR3 --> Qubit conc. (ng/ul)', 'Quantification POST-PCR3 --> DNA amount (ng)',
            'BioAnalyzer / TS --> Peak (bp)']

for col in cols2change:
    qc_metrics[col] = qc_metrics[col].astype(float)
    
    
# Add name to nameless project

qc_metrics['PROJECT'] = qc_metrics['PROJECT'].fillna('ad_hoc')

# Fix molarity outlier (cap at 500)

qc_metrics['Molarity'] = qc_metrics['Molarity'].apply(lambda x:500 if x > 500 else x) 

# FIX missing post-PCR1 metrics for IDT multiplexed

columns_to_fill = ['Real input to capture --> DNA amount (ng)',
'Quantification POST-PCR2 --> Qubit conc. (ng/ul)',
'Quantification POST-PCR2 --> DNA amount (ng)',
'TapeStation after PCR2 --> Size (bp)',
'Quantification POST-PCR3 --> Qubit conc. (ng/ul)',
'Quantification POST-PCR3 --> DNA amount (ng)',
'BioAnalyzer / TS --> Peak (bp)',
'Molarity']

for ind, row in qc_metrics.iterrows():
    
    if pd.notna(row['Multiplexing']):
        sample_multi = row['Multiplexing']
        # Get the matching row(s) from the DataFrame
        row_multi = qc_metrics[qc_metrics['Sample ID'] == sample_multi]

        # If a matching row is found
        if not row_multi.empty:
            for column in columns_to_fill:
                # Use .iloc[0] in case there are multiple matches
                qc_metrics.loc[ind, column] = row_multi.iloc[0][column]
                
# Create yes/no multiplexing column

qc_metrics['Multiplexing_binary'] = qc_metrics['Multiplexing'].notna().map({True: 'Yes', False: 'No'})
        
# Add information on single or double capture

qc_metrics["NUM_CAPTURES"] = 0

# qc_metrics[(qc_metrics['Quantification POST-PCR3 --> DNA amount (ng)'].isna())
#            & (qc_metrics['Quantification POST-PCR2 --> DNA amount (ng)'].isna())
#           ]["NUM_CAPTURES"] = 0

qc_metrics.loc[(qc_metrics['Quantification POST-PCR3 --> DNA amount (ng)'].isna())
            & (~qc_metrics['Quantification POST-PCR2 --> DNA amount (ng)'].isna())
            ,"NUM_CAPTURES"
            ] = 1

qc_metrics.loc[(~qc_metrics['Quantification POST-PCR3 --> DNA amount (ng)'].isna())
            & (~qc_metrics['Quantification POST-PCR2 --> DNA amount (ng)'].isna())
            ,"NUM_CAPTURES"
            ] = 2

# Add information on splitted PCR1 output before capture

qc_metrics["SPLIT_PCR"] = ~(qc_metrics['Quantification POST-PCR1 --> DNA amount (ng)'] == qc_metrics['Real input to capture --> DNA amount (ng)'])
qc_metrics["SUBSET_DIVERSITY"] = ~(qc_metrics["qPCR --> Unique DNA amount (fmol)"] == qc_metrics['Real diversity --> fmol to PCR1'])

# # Add information on whether it has been sequenced or not

# qc_metrics["Sequenced"] = ~qc_metrics["IRB_subsample_id"].isna()

## Summarize information on library size
# A single column with all the values, no matter the number of captures nor anything

lib_size_values = []
for ind, row in qc_metrics.iterrows():
    value = np.nan
    for col in ['BioAnalyzer / TS --> Peak (bp)',
                'TapeStation after PCR2 --> Size (bp)',
                'TapeStation after PCR1 --> Size (bp)',
                'TapeStation (after ligation) --> Size (bp)',
                'TapeStation (after fragmentation) --> Size (bp)']:

        if not np.isnan(row[col]):
            value = row[col]
            break

    lib_size_values.append(value)
    
qc_metrics['Library_size (bp)'] = lib_size_values

# Add information on library and ligation sizes without adapters

# adapter_size = 150
qc_metrics["Ligation nonadapter size (bp)"] = qc_metrics["TapeStation (after ligation) --> Size (bp)"] - adapter_size

# adapter_index_size = 180
qc_metrics['Library nonadapter size (bp)'] = qc_metrics['Library_size (bp)'] - adapter_index_size

### RECOVERY metrics
qc_metrics["Recovery_Input2Lig_raw"] = qc_metrics["Quantification (after ligation) --> DNA amount (ng)"] / qc_metrics["Input (ng)"]

# qc_metrics["Lig_genomic_DNA"] = qc_metrics["Quantification (after ligation) --> DNA amount (ng)"]  \
#                                     * (qc_metrics["Ligation nonadapter size (bp)"] / qc_metrics["TapeStation (after ligation) --> Size (bp)"])

# qc_metrics["Recovery_Input2Lig_genomic"] = qc_metrics["Lig_genomic_DNA"] / qc_metrics["Input (ng)"]

qc_metrics["qPCR unique molecules"] = qc_metrics['qPCR --> Unique DNA amount (fmol)'] * avogadro_in_f_scale

qc_metrics["Lig_qpcr_DNA"] = qc_metrics["qPCR unique molecules"] * \
                                            qc_metrics["Ligation nonadapter size (bp)"] * bp_weight_in_ng
                                            
qc_metrics["Recovery_Input2Lig_qpcr"] = qc_metrics["Lig_qpcr_DNA"] / qc_metrics["Input (ng)"]

# Compute difference between ligation and PCR peaks

qc_metrics["TapeStation_PCR1-Ligation"] = qc_metrics['TapeStation after PCR1 --> Size (bp)'] \
                                                - qc_metrics['TapeStation (after ligation) --> Size (bp)']

### Compute EXPECTED DEPTH from wet lab molecules
# max_contribution_per_read = 300 - 18

qc_metrics["Effective_bp_per_molecule-lig"] = qc_metrics["Ligation nonadapter size (bp)"].apply(lambda x : min(max_contribution_per_read, x) )
qc_metrics["Effective_bp_per_molecule-lib"] = qc_metrics['Library nonadapter size (bp)'].apply(lambda x : min(max_contribution_per_read, x) )

qc_metrics["Theoretical_max_depth-lig"] = qc_metrics["qPCR unique molecules"] * qc_metrics["Effective_bp_per_molecule-lig"] / genome_size
qc_metrics["Theoretical_max_depth-lib"] = qc_metrics["qPCR unique molecules"] * qc_metrics["Effective_bp_per_molecule-lib"] / genome_size

# Order columns 
qc_metrics = qc_metrics[['PROJECT', 'Sample ID', 'used_IRB_subsample_id'] + [x for x in qc_metrics.columns if x not in ['Sample ID', 'used_IRB_subsample_id', 'PROJECT']]]
qc_metrics.columns = ["PROJECT", "Sample ID", "IRB_subsample_id"] + [x for x in qc_metrics.columns[3:] ]

# Add prefix to all columns except the first three
qc_metrics.columns = ["PROJECT", "Sample ID", "IRB_subsample_id"] + [f'WetLab>>{x}' for x in qc_metrics.columns[3:] ]
